# Manifold geometry of the FINAL CLIP embeddings

The clean linearity readout: PCA vs MLLE on **only the output embeddings**, coloured by
**class** (image tower). Same k-sweep, 2D + 3D, shared axes. Plus a **cross-modal** cell
that co-embeds image-final + text-final per model (the CLIP modality gap).

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "2")

import numpy as np
import matplotlib.pyplot as plt
from utils.scripts import manifold_viz as mv

INPUT_DIR    = "output_dir/activations_and_datasets_idxs_69"
DATASET      = "imagenet"
SEED         = 69
MODELS       = ["ViT-B-16", "ViT-B-32", "ViT-L-14", "RN50"]
TOWERS       = ["image", "text"]
K_SWEEP      = [5, 10, 20, 50, 100]
N_COMP       = 3
FINAL_BUDGET = 2000
GIF_K        = 20
GIF_DIR      = "logs/manifold_gifs_final"
RNG          = 1
print("config ready")

config ready


In [ ]:
def compute_final(model, tower):
    _, _, final, labels = mv.load_atoms(INPUT_DIR, DATASET, model, seed=SEED,
                                        tower=tower, components="all", last_n_layers=1)
    Ff, idx = mv.subsample_rows(final, FINAL_BUDGET, seed=RNG)
    lab = None if labels is None else labels[idx]
    out = {"labels": lab, "n": len(Ff)}
    p = mv.fit_pca(Ff, N_COMP)
    out["pca"] = p.transform(Ff); out["evr"] = p.explained_variance_ratio_
    out["mlle"] = {k: mv.fit_mlle(Ff, k, N_COMP, seed=RNG).transform(Ff) for k in K_SWEEP}
    return out

RESULTS = {}
for model in MODELS:
    for tower in TOWERS:
        print(f"fitting {model:10s} {tower:5s} ...", flush=True)
        RESULTS[(model, tower)] = compute_final(model, tower)
print("done")

fitting ViT-B-16   image ...
fitting ViT-B-16   text  ...
fitting ViT-B-32   image ...
fitting ViT-B-32   text  ...
fitting ViT-L-14   image ...
fitting ViT-L-14   text  ...
fitting RN50       image ...


## 1 · 2D views — final embeddings (PCA vs MLLE across k)
Image tower coloured by class label; text tower by index. Shared axes per (model, tower).

In [ ]:
def plot_final_2d(model, tower):
    r = RESULTS[(model, tower)]
    panels = [("PCA", r["pca"])] + [(f"MLLE k={k}", r["mlle"][k]) for k in K_SWEEP]
    lims = mv.shared_limits([e[:, :2] for _, e in panels])
    col = r["labels"] if r["labels"] is not None else np.arange(r["n"])
    cmap = "tab20" if r["labels"] is not None else "viridis"
    fig, axes = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3.4))
    for ax, (name, emb) in zip(axes, panels):
        sc = mv.scatter_2d(ax, emb[:, :2], col, limits=lims, title=name, cmap=cmap, s=6, alpha=0.6)
        if name == "PCA":
            evr = r["evr"]
            ax.set_xlabel(f"PC1 ({evr[0]*100:.1f}%)", fontsize=8)
            ax.set_ylabel(f"PC2 ({evr[1]*100:.1f}%)", fontsize=8)
    fig.suptitle(f"{model} · {tower} tower · final embeddings (n={r['n']})", fontsize=11)
    plt.show()

for model in MODELS:
    for tower in TOWERS:
        plot_final_2d(model, tower)

## 2 · 3D rotating GIFs — final embeddings
PCA and MLLE (k=`GIF_K`) per (model, tower).

In [ ]:
from IPython.display import Image, display
gif_paths = []
for model in MODELS:
    for tower in TOWERS:
        r = RESULTS[(model, tower)]
        col = r["labels"] if r["labels"] is not None else np.arange(r["n"])
        cmap = "tab20" if r["labels"] is not None else "viridis"
        for name, emb in [("pca", r["pca"]), (f"mlle_k{GIF_K}", r["mlle"][GIF_K])]:
            p = f"{GIF_DIR}/{model}_{tower}_{name}.gif"
            mv.rotate_gif(emb, col, p, cmap=cmap, title=f"{model} {tower} {name}")
            gif_paths.append(p); print("saved", p, flush=True)
for p in gif_paths:
    display(Image(filename=p))

## 3 · Cross-modal — image-final + text-final co-embedded (the modality gap)
Coloured by modality; PCA and MLLE (k=`GIF_K`).

In [ ]:
fig, axes = plt.subplots(2, len(MODELS), figsize=(3.4 * len(MODELS), 6.6))
for j, model in enumerate(MODELS):
    _, _, img_f, _ = mv.load_atoms(INPUT_DIR, DATASET, model, seed=SEED, tower="image", last_n_layers=1)
    _, _, txt_f, _ = mv.load_atoms(INPUT_DIR, DATASET, model, seed=SEED, tower="text",  last_n_layers=1)
    img_f, _ = mv.subsample_rows(img_f, 1200, seed=RNG)
    txt_f, _ = mv.subsample_rows(txt_f, 1000, seed=RNG)
    stack = np.concatenate([img_f, txt_f], 0)
    mod = np.concatenate([np.zeros(len(img_f)), np.ones(len(txt_f))])
    pca = mv.fit_pca(stack, N_COMP).transform(stack)
    mlle = mv.fit_mlle(stack, GIF_K, N_COMP, seed=RNG).transform(stack)
    for i, (name, emb) in enumerate([("PCA", pca), (f"MLLE k={GIF_K}", mlle)]):
        ax = axes[i, j]
        ax.scatter(emb[mod == 0, 0], emb[mod == 0, 1], s=5, alpha=0.5, label="image", c="tab:blue")
        ax.scatter(emb[mod == 1, 0], emb[mod == 1, 1], s=5, alpha=0.5, label="text",  c="tab:red")
        ax.set_title(f"{model} · {name}", fontsize=9)
        if i == 0 and j == 0:
            ax.legend(fontsize=7)
fig.suptitle("Cross-modal geometry: image vs text final embeddings", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# scree + Procrustes(PCA,MLLE) for the final embeddings
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for model in MODELS:
    for tower in TOWERS:
        _, _, final, _ = mv.load_atoms(INPUT_DIR, DATASET, model, seed=SEED, tower=tower, last_n_layers=1)
        Ff, _ = mv.subsample_rows(final, FINAL_BUDGET, seed=RNG)
        cum = np.cumsum(mv.fit_pca(Ff, 30).explained_variance_ratio_)
        axes[0].plot(range(1, 31), cum, marker=".", label=f"{model}/{tower}")
        r = RESULTS[(model, tower)]
        disp = [mv.procrustes_score(r["pca"], r["mlle"][k]) for k in K_SWEEP]
        axes[1].plot(K_SWEEP, disp, marker="o", label=f"{model}/{tower}")
axes[0].axhline(0.9, ls="--", c="grey", lw=0.8); axes[0].set_xlabel("n components")
axes[0].set_ylabel("cumulative variance"); axes[0].set_title("PCA scree (final embeddings)")
axes[1].set_xscale("log"); axes[1].set_xlabel("MLLE k"); axes[1].grid(alpha=0.3)
axes[1].set_ylabel("Procrustes disparity"); axes[1].set_title("Procrustes(PCA, MLLE) — final")
axes[1].legend(fontsize=7, ncol=2); plt.show()